# Validacion de integridad de los datos

In [ ]:
import requests #  librería necesaria para hacer peticiones web

def auditar_integridad_y_esquema_url(url_archivo):
    """
    Realiza una auditoría de un archivo remoto descargándolo en streaming para
    verificar su integridad criptográfica y validar su esquema estructural básico.

    UTILIDAD Y PROPÓSITO DE LA INTEGRIDAD (HASH SHA-256):

    1. Trazabilidad y MLOps: El hash actúa como una "huella digital" inmutable del dataset.
       Permite registrar con qué versión exacta de los datos se entrenó un modelo. Si el
       archivo en GitHub es actualizado o sobreescrito en el futuro, el hash cambiará,
       alertando al equipo sobre la modificación y garantizando la reproducibilidad.

    2. Prevención de Corrupción de Datos: Al descargar archivos de internet, existen
       riesgos de micro-cortes de red o descargas incompletas. El cálculo del hash asegura
       que los bytes procesados coinciden exactamente con la totalidad del archivo esperado,
       evitando que datos corruptos envenenen el pipeline de Machine Learning.


    UTILIDAD Y PROPÓSITO DE LA VALIDACIÓN DE ESQUEMA:

    1. Principio "Fail-Fast" (Falla Rápido): Al leer solo la primera fila del CSV (nrows=0),
       el sistema verifica la existencia de las variables críticas sin tener que cargar
       gigabytes de datos a la RAM. Si falta una columna, el proceso se detiene de
       inmediato, ahorrando tiempo de cómputo y costos de procesamiento en la nube.

    Parámetros:
    url_archivo (str): La URL directa (raw) del archivo CSV a auditar.

    Retorna:
    str: El hash SHA-256 del archivo si las validaciones son exitosas, o None si ocurre un error.
    """
    print("Iniciando Auditoría de Integridad y Esquema desde URL")

    sha256_hash = hashlib.sha256()

    # 1. VALIDACIÓN DE INTEGRIDAD (HASH) DESDE URL
    try:
        # Hacemos una petición GET a la URL.
        # stream=True permite leer el archivo por partes sin cargarlo todo en RAM
        respuesta = requests.get(url_archivo, stream=True)
        respuesta.raise_for_status() # Lanza un error si la descarga falla (ej. error 404)

        # Leemos el archivo en bloques de 4KB (4096 bytes)
        for byte_block in respuesta.iter_content(chunk_size=4096):
            if byte_block: # Filtramos los "keep-alive" vacíos
                sha256_hash.update(byte_block)

        checksum = sha256_hash.hexdigest()

        print("INTEGRIDAD: Checksum SHA-256 calculado correctamente.")
        print(f"   -> Archivo: {url_archivo}")
        print(f"   -> Hash: {checksum}")

    except requests.exceptions.RequestException as e:
        print(f"ERROR: No se pudo descargar el archivo de la URL. Detalle: {e}")
        return None

    # 2. VALIDACIÓN DE ESQUEMA (COLUMNAS) DESDE URL
    try:
        # Pandas puede leer URLs directamente. Mantenemos nrows=0 para mayor velocidad.
        df_esquema = pd.read_csv(url_archivo, nrows=0)

        columnas_encontradas = df_esquema.columns.tolist()
        columnas_criticas = ['id_cliente', 'abandono', 'ingreso_mensual', 'deuda_total']

        columnas_faltantes = [col for col in columnas_criticas if col not in columnas_encontradas]

        if not columnas_faltantes:
            print("ESQUEMA: Todas las variables críticas están presentes en el origen.")
        else:
            print(f"ADVERTENCIA: Faltan las siguientes columnas críticas: {columnas_faltantes}")

    except Exception as e:
        print(f"ERROR al validar el esquema con Pandas: {e}")
        return None

    return checksum

# --- CÓMO USARLO ---
url = 'https://raw.githubusercontent.com/agustilin/Ev1_grupo_5/refs/heads/main/data/dataset_clientes.csv'

# Le pasamos la variable 'url'
hash_dataset = auditar_integridad_y_esquema_url(url)

Iniciando Auditoría de Integridad y Esquema desde URL
INTEGRIDAD: Checksum SHA-256 calculado correctamente.
   -> Archivo: https://raw.githubusercontent.com/agustilin/Ev1_grupo_5/refs/heads/main/data/dataset_clientes.csv
   -> Hash: ac8ed445faf148dac481236a2c045716ef0a6d6f54f6f8fbab6d8c5efc7ecfac
ESQUEMA: Todas las variables críticas están presentes en el origen.
